# tw-market-data-lake — Colab Runner (thin runner)

只負責：參數 → 掛載 Drive → Secrets → clone/pull → 安裝 → Preflight → 執行 → 顯示結果。
抓取／驗證／寫入邏輯全部在 GitHub `src/tw_lake/`（Single Source of Truth）。

**建議順序**：`smoke` → 把 capability_report 結果確認後 → `pilot` → `full --dry-run` → `full`（`CONFIRM_FULL=True`）→ 之後定期 `update`。

- 只有這個 repo 擁有的資料夾會被寫入；`finlab_0050`、`fubon_txf`、`IKBR` 等既有資料夾不會被讀寫。
- 同一時間只能有一個 writer（Drive 上的 lock）。

In [ ]:
# 1. 參數
MODE = "smoke"            # smoke | pilot | full | update
DRY_RUN = False           # True: 只估算請求數與時間，不寫任何資料集/manifest
CONFIRM_FULL = False      # full 模式必須明確設 True
SYMBOLS = ""              # 例 "2330,0050"；空白 = 依模式預設
MAX_REQUESTS = None       # 例 500；None = 不限
BREAK_STALE_LOCK = False  # 只有確認前一個 run 已死才設 True
RETRY_CONFLICTS = False
RETRY_EXHAUSTED = False
DEEP_VERIFY = False       # True: 對所有已發布檔案做 SHA256 驗證（較慢）

REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
FUBON_PFX_DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/Fubon_Cert/your_cert.pfx"   # 改成實際位置
FUBON_SDK_PATH = ""       # 選填：Drive 上的富邦 Linux SDK .zip/.whl；空白則自官方網址下載
PROJECT_DIR = f"/content/{REPO_NAME}"

In [ ]:
# 2. 掛載 Google Drive + 載入 Colab Secrets（值不會被印出）
from google.colab import drive, userdata
import os
drive.mount("/content/drive")

def secret(name, required=True):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    if required and not v:
        raise RuntimeError(f"缺少 Colab Secret: {name}")
    return v or ""

os.environ["FINLAB_API_TOKEN"] = secret("FINLAB_API_TOKEN")
os.environ["FUBON_USER_ID"] = secret("FUBON_USER_ID")
os.environ["FUBON_PASSWORD"] = secret("FUBON_PASSWORD", required=False)
os.environ["FUBON_CERT_PASSWORD"] = secret("FUBON_CERT_PASSWORD", required=False)
os.environ["FUBON_API_KEY"] = secret("FUBON_API_KEY", required=False)
if not (os.environ["FUBON_PASSWORD"] or os.environ["FUBON_API_KEY"]):
    raise RuntimeError("需要 FUBON_PASSWORD 或 FUBON_API_KEY 其中之一")
os.environ["FUBON_CERT_PATH"] = FUBON_PFX_DRIVE_PATH
os.environ["DATA_ROOT"] = DATA_ROOT
os.environ["TW_LAKE_LOCAL_TMP"] = "/content/tmp_tw_lake"
if FUBON_SDK_PATH:
    os.environ["FUBON_SDK_PATH"] = FUBON_SDK_PATH
print("[OK] Drive mounted; secrets loaded (values hidden)")

In [ ]:
# 3. clone / pull 指定 branch，記錄 HEAD SHA
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
SHA = subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip()
os.environ["TW_LAKE_GIT_SHA"] = SHA
print("branch:", BRANCH, "HEAD:", SHA)

In [ ]:
# 4. 安裝依賴 + 富邦 Linux SDK
import subprocess, sys
subprocess.run([sys.executable, f"{PROJECT_DIR}/scripts/install_colab_dependencies.py"], check=True)

In [ ]:
# 5. Preflight（唯讀）
import sys, json, os
from pathlib import Path
print("Python:", sys.version.split()[0], "(target 3.11)")
root = Path(DATA_ROOT)
assert root.exists(), f"DATA_ROOT 不存在：{root}"
assert Path(FUBON_PFX_DRIVE_PATH).exists(), f"找不到 PFX：{FUBON_PFX_DRIVE_PATH}"
print("DATA_ROOT 內容：", sorted(p.name for p in root.iterdir()))
lock = root / "_tw-market-data-lake" / "locks" / "writer.lock"
print("writer lock:", lock.read_text() if lock.exists() else "none")
for d in ["finlab_stock_daily/raw", "finlab_stock_daily/adjusted", "finlab_stock_reference/universe_daily",
          "finlab_stock_reference/disposal_attention", "fubon_stock_reference/symbol_master", "fubon_stock_5m"]:
    p = root / d / "_manifest" / "published.json"
    print(f"{d:45s}", json.loads(p.read_text()) if p.exists() else "(not published yet)")

In [ ]:
# 6. 執行（log 即時串流；完整 log 會存到 run 資料夾）
import subprocess, sys
args = [sys.executable, "-m", "tw_lake.cli", "--mode", MODE]
if DRY_RUN: args.append("--dry-run")
if CONFIRM_FULL: args.append("--confirm-full")
if SYMBOLS.strip(): args += ["--symbols", SYMBOLS.strip()]
if MAX_REQUESTS: args += ["--max-requests", str(MAX_REQUESTS)]
if BREAK_STALE_LOCK: args.append("--break-stale-lock")
if RETRY_CONFLICTS: args.append("--retry-conflicts")
if RETRY_EXHAUSTED: args.append("--retry-exhausted")
if DEEP_VERIFY: args.append("--deep-verify")
print("+", " ".join(args))
proc = subprocess.Popen(args, cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
print("\nexit code:", rc, {0: "success", 2: "partial (see qa_summary)", 1: "failed / blocked"}.get(rc))

# 等 Google Drive 把本次寫入的檔案全部上傳完再結束（避免 runtime 關閉後留下 0 byte 檔案）
from google.colab import drive
drive.flush_and_unmount()
drive.mount("/content/drive")
print("[OK] Drive writes flushed and remounted")

# 重新掛載後才看得到 Drive 真正存了什麼（同一個掛載內讀到的是本機快取）：
# 逐一核對本次 run 寫入的檔案大小與 SHA256。PROBLEMS 時不要用那些分區，再跑一次 update 會自動重建。
if MODE != "smoke" and not DRY_RUN:
    vr = subprocess.run([sys.executable, "-m", "tw_lake.audit.verify", "--data-root", DATA_ROOT],
                        cwd=PROJECT_DIR, env=dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src"))
    print("verify exit code:", vr.returncode)

In [ ]:
# 7. 結果
from pathlib import Path
from IPython.display import Markdown, display
import json
runs = sorted((Path(DATA_ROOT) / "_tw-market-data-lake" / "runs").glob("*"))
run_dir = runs[-1]
print("run folder:", run_dir)
display(Markdown((run_dir / "qa_summary.md").read_text()))
if MODE == "smoke":
    reps = sorted((Path(DATA_ROOT) / "_tw-market-data-lake" / "smoke").glob(f"{run_dir.name}/capability_report.json"))
    if reps:
        print("capability report:", reps[0])
        print(json.dumps(json.loads(reps[0].read_text())["recommendations"], ensure_ascii=False, indent=2))
    else:
        print("本次 smoke 沒有產生 capability_report（請看上方 Error）")